# 5회차 실습: LU 분해는 왜 빠른가 (과정까지 눈으로)

결과만 보지 말고, **L·U가 무엇이고 / 방법 1은 뭘 반복하고 / 방법 2는 그걸 어떻게 아끼는지**를 출력으로 따라간다.

In [ ]:
import numpy as np, time
from scipy.linalg import lu, lu_factor, lu_solve
np.set_printoptions(precision=3, suppress=True)

## ① 먼저 LU 분해를 눈으로 (작은 3×3)

$A$를 **하삼각 $L$**과 **상삼각 $U$**의 곱으로 쪼갠다 ($A=LU$, 행 교환이 있으면 $A=PLU$). **이 분해가 비싼 부분**($O(n^3)$)이다.

In [ ]:
A = np.array([[2., 1., 1.],
              [4., 3., 3.],
              [8., 7., 9.]])
print('주어진 행렬 A ='); print(A)

P, L, U = lu(A)                 # A = P @ L @ U
print('\nL (하삼각, 대각 1) ='); print(L)
print('\nU (상삼각, 대각 pivot) ='); print(U)
print('\nP @ L @ U 로 A 복원 ='); print(P @ L @ U)
print('\n분해가 맞는가 (P@L@U == A):', np.allclose(P @ L @ U, A))

## ② 방법 1 vs 방법 2: 무엇을 반복하나

우변 $\mathbf{b}$가 여러 개다. 두 방법 다 **같은 답**을 주지만, 반복하는 일이 다르다.

- **방법 1** `solve(A,b)`: b마다 **L·U를 처음부터 다시 만든다** (비싼 분해를 매번 반복)
- **방법 2** `lu_factor`→`lu_solve`: **L·U를 한 번만** 만들고, b마다 **대입(전진·후진)만**

In [ ]:
bs = [np.array([1., 2., 3.]), np.array([0., 1., 0.]), np.array([5., 5., 5.])]

print('=== 방법 1: b마다 solve (매번 L,U 재계산) ===')
for i, b in enumerate(bs):
    x = np.linalg.solve(A, b)
    print(f'  b{i+1}={b}  ->  x={x}   (이 줄마다 분해를 다시 함)')

print('\n=== 방법 2: 분해는 딱 한 번, 그다음 대입만 ===')
lu_piv = lu_factor(A)                 # 분해 1회 (여기서만 O(n^3))
print('  분해 완료 -> 이제 b마다 대입만 (O(n^2))')
for i, b in enumerate(bs):
    x = lu_solve(lu_piv, b)
    print(f'  b{i+1}={b}  ->  x={x}   (같은 L,U 재사용)')

print('\n두 방법의 답이 같은가:', all(np.allclose(np.linalg.solve(A,b), lu_solve(lu_piv,b)) for b in bs))

## ③ 큰 행렬에서 시간 측정: 아낀 게 얼마인가

방법 1은 분해를 $m$번, 방법 2는 $1$번. $n$이 크면 분해($O(n^3)$)가 지배하므로 차이가 크게 벌어진다.

In [ ]:
rng = np.random.default_rng(0)
n, m = 1500, 80
Big = rng.standard_normal((n, n)) + n*np.eye(n)
B = [rng.standard_normal(n) for _ in range(m)]

t0 = time.perf_counter()
for b in B: np.linalg.solve(Big, b)          # 분해 m번
t1 = time.perf_counter() - t0

t0 = time.perf_counter()
piv = lu_factor(Big)                          # 분해 1번
for b in B: lu_solve(piv, b)                  # 대입 m번
t2 = time.perf_counter() - t0

print(f'n={n}, b {m}개')
print(f'  방법 1 (분해 {m}번) : {t1*1000:7.0f} ms')
print(f'  방법 2 (분해 1번)  : {t2*1000:7.0f} ms')
print(f'  -> 약 {t1/t2:.1f}배 빠름  (분해를 {m}번 -> 1번으로 줄인 효과)')

## 정리

- **L·U**: $A$를 삼각행렬 둘로 쪼갠 것. 만드는 데 $O(n^3)$ (비쌈).
- **방법 1**은 b마다 이 분해를 **다시** 한다 → 낭비. **방법 2**는 분해를 **재사용**하고 대입($O(n^2)$)만 → 빠름.
- 그래서 `np.linalg.solve`도 역행렬을 만들지 않고 내부에서 LU로 풀며, 같은 $A$에 b가 많으면 `lu_factor`+`lu_solve`가 정석이다.
- **LU의 진짜 목적**: 같은 $A$에 여러 $\mathbf{b}$를 빠르게 푸는 것.